In [1]:
#### python
import os
import sys
import importlib
# columnar analysis
from coffea import processor
from coffea.nanoevents import NanoAODSchema
import awkward as ak
from dask.distributed import Client, performance_report
# local
sidm_path = str(os.getcwd()).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import utilities, sidm_processor, scaleout, llpnanoaodschema
# always reload local modules to pick up changes during development
importlib.reload(utilities)
importlib.reload(sidm_processor)
importlib.reload(scaleout)
#import 
# plotting
import matplotlib.pyplot as plt
utilities.set_plot_style()
%matplotlib inline
import coffea.util
from matplotlib.colors import LogNorm
from coffea.processor import accumulate

In [2]:
# Constants
vr = "19"
npl, fw, fh = 2, 16, 9
n_scl = 1#200000
npl =2

In [3]:
# opening files
output_2mu = coffea.util.load("outputs/BKG_sig2mu_" + vr + ".coffea")
output_4mu = coffea.util.load("outputs/BKG_sig4mu_" + vr + ".coffea")
output_dyj = coffea.util.load("outputs/BKG_bkgdyj_" + vr + ".coffea")
output_ttj = coffea.util.load("outputs/BKG_bkgttj_" + vr + ".coffea")
output_qcd = coffea.util.load("outputs/BKG_bkgqcd_" + vr + ".coffea")
output_ocd = coffea.util.load("outputs/BKG_bkgocd_" + vr + ".coffea")

In [4]:
# Signal and BKG files

mxx_2mu = ["2Mu2E_200GeV_5p0GeV_100p0mm", "2Mu2E_500GeV_5p0GeV_80p0mm", "2Mu2E_800GeV_5p0GeV_50p0mm","2Mu2E_1000GeV_5p0GeV_40p0mm",]# Mxx_5_300
mzd_2mu = ["2Mu2E_500GeV_0p25GeV_4p0mm", "2Mu2E_500GeV_1p2GeV_19p0mm", "2Mu2E_500GeV_5p0GeV_80p0mm", ] # 500_Mzd_300
lxy_2mu = ["2Mu2E_500GeV_1p2GeV_0p019mm", "2Mu2E_500GeV_1p2GeV_0p19mm", "2Mu2E_500GeV_1p2GeV_1p9mm", "2Mu2E_500GeV_1p2GeV_9p6mm", "2Mu2E_500GeV_1p2GeV_19p0mm"] # 500_1.2_Lxy

mxx_4mu = [    "4Mu_200GeV_5p0GeV_200p0mm",    "4Mu_500GeV_5p0GeV_80p0mm",    "4Mu_800GeV_5p0GeV_50p0mm",    "4Mu_1000GeV_5p0GeV_40p0mm",]
mzd_4mu = [    "4Mu_500GeV_0p25GeV_0p004mm",    "4Mu_500GeV_1p2GeV_19p0mm", "4Mu_500GeV_5p0GeV_80p0mm",]
lxy_4mu = [    "4Mu_500GeV_1p2GeV_0p019mm",     "4Mu_500GeV_1p2GeV_0p19mm",    "4Mu_500GeV_1p2GeV_1p9mm",    "4Mu_500GeV_1p2GeV_9p6mm",    "4Mu_500GeV_1p2GeV_19p0mm",]

sam_4mu = mxx_4mu + mzd_4mu + lxy_4mu

bkgdyj = ["DYJetsToMuMu_M10to50",    "DYJetsToMuMu_M50",]
bkgttj = ["TTJets"]
bkgqcd = ["QCD_Pt15To20",    "QCD_Pt20To30",    "QCD_Pt30To50",    "QCD_Pt50To80",    "QCD_Pt80To120",]
bkgocd = ["QCD_Pt120To170",    "QCD_Pt170To300",    "QCD_Pt300To470",    "QCD_Pt470To600",    "QCD_Pt600To800", "QCD_Pt1000",] # "QCD_Pt800To1000", # does not work
allqcd = bkgqcd + bkgocd

In [5]:
#Variables 
# Hist_2mu = ["mulj_egmlj_invmass", "lj_lj_absdR", "lj_lj_absdeta"]


#channels: cuts to be applied (slections.yaml) 
channels = ["baseNoLj", "bkg_study_isopdisp", "bkg_study_isopdisp_2lj", 
            "bkg_study_isopdisp_2lj_dPhi_4mu", 
            
            "bkg_study_isopdisp_2lj_dPhi_2mu2e", 
            "bkg_study_isopdisp_2lj_dPhi2p2_2mu2e",
            "bkg_study_isopdisp_2lj_dPhi1p8_2mu2e"
           ]

ch1 = channels[0] #base
ch2 = channels[1] # iso + disp
ch3 = channels[2] # 2lj
ch4 = channels[3] # dphi2 4mu
ch5 = channels[4] # dphi2 2mu
ch6 = channels[5] # dphi2.2 2mu
ch7 = channels[6] # dphi2.2 2mu

all_cha = [ch1, ch2, ch3, ch4, ch5, ch6, ch7]
all_cha_names = ["base", "Iso + Disp", ">2LJ", "4mu", "2mu2e"]
col = ["r", "g", "b"]

In [27]:
for i in mzd_2mu:
    print(i)
    output_2mu[i]["cutflow"][ch5].print_table()
    print()

2Mu2E_500GeV_0p25GeV_4p0mm
cut name                 raw N    weighted N    weighted %
--------------------  --------  ------------  ------------
None                  138237.0          59.8         100.0
pass triggers          30101.0          13.0          21.8
PV filter              30101.0          13.0          21.8
>=2 LJs                10010.0           4.3           7.2
>=1 egm_ljs             8902.0           3.9           6.4
>=1 mu_ljs              7192.0           3.1           5.2
dPhi(LJ_0, LJ_1) > 2    7075.0           3.1           5.1

2Mu2E_500GeV_1p2GeV_19p0mm
cut name                 raw N    weighted N    weighted %
--------------------  --------  ------------  ------------
None                  138545.0          59.8         100.0
pass triggers          26002.0          11.2          18.8
PV filter              26002.0          11.2          18.8
>=2 LJs                 8281.0           3.6           6.0
>=1 egm_ljs             7227.0           3.1           5.2
>

In [28]:
print("\n TTJ \n")
output_ttj[bkgttj[0]]["cutflow"][ch5].print_table()


 TTJ 

cut name                  raw N    weighted N    weighted %
--------------------  ---------  ------------  ------------
None                  2396626.0     7751378.5         100.0
pass triggers          148050.0      478855.2           6.2
PV filter              148009.0      478745.2           6.2
>=2 LJs                 84491.0      273291.9           3.5
>=1 egm_ljs              1121.0        3626.0           0.0
>=1 mu_ljs                 54.0         174.7           0.0
dPhi(LJ_0, LJ_1) > 2       29.0          93.8           0.0


In [30]:
for i in bkgdyj:
    print(i)
    output_dyj[i]["cutflow"][ch5].print_table()


DYJetsToMuMu_M10to50
cut name                 raw N    weighted N    weighted %
--------------------  --------  ------------  ------------
None                  121975.0     5532717.0         100.0
pass triggers          14606.0      642088.6          11.6
PV filter              14606.0      642088.6          11.6
>=2 LJs                 1516.0       69574.3           1.3
>=1 egm_ljs               25.0        1146.4           0.0
>=1 mu_ljs                 0.0           0.0           0.0
dPhi(LJ_0, LJ_1) > 2       0.0           0.0           0.0
DYJetsToMuMu_M50
cut name                    raw N    weighted N    weighted %
--------------------  -----------  ------------  ------------
None                  110516373.0    70293528.0         100.0
pass triggers          55283948.0       56522.1           0.1
PV filter              55283930.0       56522.0           0.1
>=2 LJs                44243036.0       44829.6           0.1
>=1 egm_ljs               44185.0          48.4           0

In [31]:
for i in bkgqcd:
    print(i)
    output_qcd[i]["cutflow"][ch5].print_table()
    print("\n")

for i in bkgocd:
    print(i)
    output_ocd[i]["cutflow"][ch5].print_table()
    print("\n")

QCD_Pt15To20
cut name                raw N    weighted N    weighted %
--------------------  -------  ------------  ------------
None                    853.0    22589406.0         100.0
pass triggers           185.0     4785097.0          21.2
PV filter               179.0     4785097.0          21.2
>=2 LJs                   0.0           0.0           0.0
>=1 egm_ljs               0.0           0.0           0.0
>=1 mu_ljs                0.0           0.0           0.0
dPhi(LJ_0, LJ_1) > 2      0.0           0.0           0.0


QCD_Pt20To30
cut name                raw N    weighted N    weighted %
--------------------  -------  ------------  ------------
None                  22954.0    66215060.0         100.0
pass triggers          1833.0     5287562.5           8.0
PV filter              1833.0     5287562.5           8.0
>=2 LJs                   7.0       20192.5           0.0
>=1 egm_ljs               0.0           0.0           0.0
>=1 mu_ljs                0.0           0.0 

In [ ]:
raise SystemExit("Notebook stopped intentionally after this cell.")